# 01 — PySpark Telemetry Time Buckets

Model playground notebook parallel to the Pandas version.


## Cell 2 — Imports and Spark session


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Pull PostgreSQL JDBC driver at session start so Spark can read Postgres tables.
spark = (
    SparkSession.builder
    .appName('telemetry-time-buckets-playground')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.warehouse.dir', '/workspace/data/spark-warehouse')
    .config('spark.driver.host', '127.0.0.1')
    .config('spark.driver.bindAddress', '127.0.0.1')
    .config('spark.jars.packages', 'org.postgresql:postgresql:42.7.4')
    .getOrCreate()
)
print('Spark version:', spark.version)


Spark version: 3.5.4


26/05/14 09:29:43 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Cell 3 — PostgreSQL connection settings (for JDBC)


In [8]:
DB_HOST = 'host.docker.internal'
DB_PORT = '5432'
DB_NAME = 'observability'
DB_USER = 'obs_user'
DB_PASS = 'obs_pass'
JDBC_URL = f'jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}'
JDBC_PROPS = {
    'user': DB_USER,
    'password': DB_PASS,
    'driver': 'org.postgresql.Driver'
}
JDBC_URL


'jdbc:postgresql://host.docker.internal:5432/observability'

## Cell 4 — Load telemetry table (JDBC)


In [9]:
# If this fails with ClassNotFoundException for org.postgresql.Driver,
# we will add the PostgreSQL JDBC jar in the Docker image.
df = (
    spark.read
    .format('jdbc')
    .option('url', JDBC_URL)
    .option('dbtable', 'lab.telemetry_cpu_raw')
    .option('user', DB_USER)
    .option('password', DB_PASS)
    .option('driver', 'org.postgresql.Driver')
    .load()
)
df.printSchema()
df.show(5, truncate=False)


root
 |-- id: long (nullable = true)
 |-- sampled_at: timestamp (nullable = true)
 |-- host: string (nullable = true)
 |-- cpu_pct: decimal(5,2) (nullable = true)
 |-- mem_pct: decimal(5,2) (nullable = true)
 |-- region: string (nullable = true)
 |-- env: string (nullable = true)



+---+--------------------------+------+-------+-------+---------+-----+
|id |sampled_at                |host  |cpu_pct|mem_pct|region   |env  |
+---+--------------------------+------+-------+-------+---------+-----+
|1  |2026-04-01 00:00:58.097654|host61|83.15  |80.41  |eu-west-1|stage|
|2  |2026-04-01 00:00:53.671443|host62|30.69  |76.73  |eu-west-1|prod |
|3  |2026-04-01 00:00:51.986285|host45|65.21  |87.93  |us-east-1|prod |
|4  |2026-04-01 00:01:12.12087 |host04|45.01  |74.16  |eu-west-1|stage|
|5  |2026-04-01 00:01:14.956693|host56|79.79  |50.04  |eu-west-1|prod |
+---+--------------------------+------+-------+-------+---------+-----+
only showing top 5 rows



## Cell 5 — Helper: quick table stats


In [10]:
def show_stats(input_df):
    input_df.agg(
        F.count('*').alias('rows'),
        F.min('sampled_at').alias('min_ts'),
        F.max('sampled_at').alias('max_ts'),
        F.countDistinct('host').alias('hosts')
    ).show(truncate=False)

show_stats(df)


+------+--------------------------+--------------------------+-----+
|rows  |min_ts                    |max_ts                    |hosts|
+------+--------------------------+--------------------------+-----+
|371521|2026-04-01 00:00:51.986285|2026-05-14 00:00:43.774949|80   |
+------+--------------------------+--------------------------+-----+



## Cell 6 — Hourly bucket aggregation


In [11]:
hourly = (
    df.where(F.col('env') == 'prod')
      .groupBy(F.date_trunc('hour', F.col('sampled_at')).alias('hour_bucket'), 'host')
      .agg(
          F.round(F.avg('cpu_pct'), 2).alias('avg_cpu'),
          F.max('cpu_pct').alias('peak_cpu')
      )
      .orderBy(F.col('hour_bucket').desc(), F.col('host'))
)
hourly.show(25, truncate=False)


+-------------------+------+-------+--------+
|hour_bucket        |host  |avg_cpu|peak_cpu|
+-------------------+------+-------+--------+
|2026-05-14 00:00:00|host45|86.89  |86.89   |
|2026-05-13 23:00:00|host02|75.85  |91.69   |
|2026-05-13 23:00:00|host03|71.60  |94.64   |
|2026-05-13 23:00:00|host04|65.25  |79.62   |
|2026-05-13 23:00:00|host06|74.00  |76.69   |
|2026-05-13 23:00:00|host07|63.75  |75.88   |
|2026-05-13 23:00:00|host08|32.31  |32.31   |
|2026-05-13 23:00:00|host09|48.57  |63.31   |
|2026-05-13 23:00:00|host10|45.09  |55.19   |
|2026-05-13 23:00:00|host12|41.85  |53.83   |
|2026-05-13 23:00:00|host13|59.53  |88.23   |
|2026-05-13 23:00:00|host16|67.58  |80.03   |
|2026-05-13 23:00:00|host17|51.83  |83.22   |
|2026-05-13 23:00:00|host18|71.62  |87.75   |
|2026-05-13 23:00:00|host19|46.32  |76.70   |
|2026-05-13 23:00:00|host20|67.01  |67.01   |
|2026-05-13 23:00:00|host22|50.34  |57.67   |
|2026-05-13 23:00:00|host23|56.17  |56.17   |
|2026-05-13 23:00:00|host24|48.35 

## Cell 7 — Rolling 6-hour average window


In [12]:
hourly_base = (
    df.groupBy(F.date_trunc('hour', F.col('sampled_at')).alias('hour_bucket'), 'host')
      .agg(F.avg('cpu_pct').alias('avg_cpu'))
)

w = Window.partitionBy('host').orderBy('hour_bucket').rowsBetween(-5, 0)

rolling = (
    hourly_base
    .withColumn('rolling_6h_avg', F.round(F.avg('avg_cpu').over(w), 2))
    .withColumn('avg_cpu', F.round(F.col('avg_cpu'), 2))
    .orderBy('host', 'hour_bucket')
)
rolling.show(50, truncate=False)


+-------------------+------+-------+--------------+
|hour_bucket        |host  |avg_cpu|rolling_6h_avg|
+-------------------+------+-------+--------------+
|2026-04-01 00:00:00|host01|57.75  |57.75         |
|2026-04-01 01:00:00|host01|64.94  |61.35         |
|2026-04-01 02:00:00|host01|54.14  |58.94         |
|2026-04-01 03:00:00|host01|71.78  |62.15         |
|2026-04-01 04:00:00|host01|39.81  |57.68         |
|2026-04-01 05:00:00|host01|63.94  |58.73         |
|2026-04-01 06:00:00|host01|66.30  |60.15         |
|2026-04-01 07:00:00|host01|49.12  |57.51         |
|2026-04-01 08:00:00|host01|65.24  |59.36         |
|2026-04-01 09:00:00|host01|75.88  |60.05         |
|2026-04-01 10:00:00|host01|54.22  |62.45         |
|2026-04-01 11:00:00|host01|78.09  |64.81         |
|2026-04-01 12:00:00|host01|67.26  |64.97         |
|2026-04-01 13:00:00|host01|82.77  |70.58         |
|2026-04-01 14:00:00|host01|56.94  |69.19         |
|2026-04-01 15:00:00|host01|56.89  |66.03         |
|2026-04-01 